## Example 1: blosc functionality test 1

In [ ]:
import blosc
import numpy as np

# Example 1: Basic compression and decompression with Blosc
print("Example 1: Basic compression and decompression with Blosc")

try:
    # Create sample data to compress
    data = b"Hello World! " * 1000  # Repetitive data compresses well
    
    print("✓ Blosc compression library imported")
    print(f"Original data size: {len(data)} bytes")
    print(f"Data sample: {data[:50]}...")
    
    # Basic compression
    compressed = blosc.compress(data)
    print(f"✓ Data compressed successfully")
    print(f"Compressed size: {len(compressed)} bytes")
    print(f"Compression ratio: {len(data) / len(compressed):.2f}:1")
    
    # Basic decompression
    decompressed = blosc.decompress(compressed)
    print(f"✓ Data decompressed successfully")
    print(f"Decompressed size: {len(decompressed)} bytes")
    
    # Verify data integrity: a lossy round-trip must fail the cell, not print and carry on
    assert decompressed == data, f"blosc round-trip lost data: {len(decompressed)} bytes back for {len(data)} in"
    print("✓ Data integrity verified - original and decompressed data match")
    
    # Show Blosc version and info (using correct attributes)
    print(f"\nBlosc information:")
    print(f"Blosc version: {blosc.__version__}")
    print(f"Blosc library: {blosc.VERSION_STRING}")
    print(f"Blosc date: {blosc.VERSION_DATE}")
    
    # Available compressors (using correct function name)
    compressor_list = blosc.compressor_list()
    print(f"Available compressors: {compressor_list}")
    assert {'blosclz', 'lz4', 'lz4hc', 'zlib', 'zstd'} <= set(compressor_list), compressor_list
    
    # Test with numerical data
    print(f"\nTesting with numerical data:")
    numeric_data = np.arange(10000, dtype=np.float64)
    numeric_bytes = numeric_data.tobytes()
    
    numeric_compressed = blosc.compress(numeric_bytes)
    numeric_ratio = len(numeric_bytes) / len(numeric_compressed)
    assert blosc.decompress(numeric_compressed) == numeric_bytes, "blosc numeric round-trip lost data"
    
    print(f"Numeric data original: {len(numeric_bytes)} bytes")
    print(f"Numeric data compressed: {len(numeric_compressed)} bytes")
    print(f"Numeric compression ratio: {numeric_ratio:.2f}:1")
    
    print("✓ Basic compression and decompression completed")
    print("blosc example 1 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"blosc example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: blosc functionality test 2

In [ ]:
import blosc
import numpy as np
import time

# Example 2: Compression algorithms and performance comparison
print("Example 2: Compression algorithms and performance comparison")

try:
    # Create test data - mix of patterns for comprehensive testing
    test_data_sets = {
        "repetitive": b"ABCDEFGH" * 10000,  # Very compressible
        "numeric_sequence": np.arange(50000, dtype=np.int32).tobytes(),  # Somewhat compressible
        "random": np.random.bytes(100000),  # Not very compressible
    }
    
    print("Testing different compression algorithms:")
    
    # Available compressors (using correct function name)
    compressors = blosc.compressor_list()
    print(f"Available compressors: {compressors}")
    
    # Test each compressor on each data type
    results = {}
    
    for data_name, data in test_data_sets.items():
        print(f"\nTesting with {data_name} data ({len(data)} bytes):")
        results[data_name] = {}
        
        for compressor in compressors:  # Round-trip every available compressor
            try:
                # Measure compression time and ratio
                start_time = time.time()
                compressed = blosc.compress(data, cname=compressor)
                compress_time = time.time() - start_time
                
                # Measure decompression time
                start_time = time.time()
                decompressed = blosc.decompress(compressed)
                decompress_time = time.time() - start_time
                
                # Calculate metrics
                compression_ratio = len(data) / len(compressed)
                
                # Verify data integrity
                integrity_ok = data == decompressed
                assert integrity_ok, f"{compressor} round-trip of {data_name} data lost data"
                
                results[data_name][compressor] = {
                    'ratio': compression_ratio,
                    'compress_time': compress_time * 1000,  # ms
                    'decompress_time': decompress_time * 1000,  # ms
                    'integrity': integrity_ok
                }
                
                print(f"  {compressor:>12}: {compression_ratio:6.2f}:1 ratio, "
                      f"C:{compress_time*1000:5.1f}ms, D:{decompress_time*1000:5.1f}ms, "
                      f"OK:{integrity_ok}")
                
            except Exception as e:
                print(f"  {compressor:>12}: Error - {type(e).__name__}")
                raise
                
    # Compression level testing
    print(f"\nCompression level testing (using lz4 if available):")
    test_compressor = 'lz4' if 'lz4' in compressors else compressors[0]
    test_data = test_data_sets['numeric_sequence']
    
    for clevel in range(0, 10, 2):  # Test levels 0, 2, 4, 6, 8
        try:
            start_time = time.time()
            compressed = blosc.compress(test_data, clevel=clevel, cname=test_compressor)
            compress_time = time.time() - start_time
            
            ratio = len(test_data) / len(compressed)
            assert blosc.decompress(compressed) == test_data, f"{test_compressor} level {clevel} round-trip lost data"
            print(f"  Level {clevel}: {ratio:6.2f}:1 ratio, {compress_time*1000:5.1f}ms")
            
        except Exception as e:
            print(f"  Level {clevel}: Error - {type(e).__name__}")
            raise
    
    # lz4hc is the high-compression LZ4 codec; check it on its own, at its maximum level
    lz4hc_data = test_data_sets['numeric_sequence']
    lz4hc_compressed = blosc.compress(lz4hc_data, typesize=4, clevel=9, cname='lz4hc')
    assert blosc.get_clib(lz4hc_compressed) == 'LZ4', blosc.get_clib(lz4hc_compressed)
    assert blosc.decompress(lz4hc_compressed) == lz4hc_data, "lz4hc round-trip lost data"

    # Best performers summary
    print(f"\nPerformance summary:")
    
    for data_name, compressor_results in results.items():
        if compressor_results:
            # Find best compression ratio
            best_ratio = max(compressor_results.items(), 
                           key=lambda x: x[1]['ratio'] if x[1]['integrity'] else 0)
            
            # Find fastest compression
            best_speed = min(compressor_results.items(),
                           key=lambda x: x[1]['compress_time'] if x[1]['integrity'] else float('inf'))
            
            print(f"  {data_name}:")
            print(f"    Best ratio: {best_ratio[0]} ({best_ratio[1]['ratio']:.2f}:1)")
            print(f"    Fastest: {best_speed[0]} ({best_speed[1]['compress_time']:.1f}ms)")
    
    print("✓ Compression algorithms and performance comparison completed")
    print("blosc example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"blosc example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: blosc functionality test 3

In [ ]:
import blosc
import numpy as np

# Example 3: Advanced Blosc features and chunking
print("Example 3: Advanced Blosc features and chunking")

try:
    print("Testing advanced Blosc features:")
    
    # Create large array for chunking demonstration
    large_array = np.random.rand(100000).astype(np.float32)
    array_bytes = large_array.tobytes()
    
    print(f"Large array size: {len(array_bytes)} bytes")
    print(f"Array shape: {large_array.shape}")
    print(f"Array dtype: {large_array.dtype}")
    
    # Chunking and multithreading
    print(f"\nTesting chunking and multithreading:")
    
    # Set number of threads
    blosc.set_nthreads(2)  # Use 2 threads
    nthreads = blosc.nthreads
    assert nthreads == 2, nthreads
    print(f"Blosc threads: {nthreads}")
    
    # Different chunk sizes
    chunk_sizes = [0, 16384, 65536, 262144]  # 0 = automatic, then 16KB, 64KB, 256KB
    
    for chunk_size in chunk_sizes:
        try:
            if chunk_size == 0:
                # Automatic chunking
                compressed = blosc.compress(array_bytes, typesize=4, shuffle=blosc.SHUFFLE)
                chunk_desc = "automatic"
            else:
                # Manual chunking (using blocksize parameter correctly)
                compressed = blosc.compress(array_bytes, typesize=4, shuffle=blosc.SHUFFLE)
                blosc.set_blocksize(chunk_size)  # Set global blocksize
                compressed = blosc.compress(array_bytes, typesize=4, shuffle=blosc.SHUFFLE)
                chunk_desc = f"{chunk_size//1024}KB"
            
            ratio = len(array_bytes) / len(compressed)
            assert blosc.decompress(compressed) == array_bytes, f"round-trip at chunk size {chunk_desc} lost data"
            print(f"  Chunk size {chunk_desc:>10}: {ratio:6.2f}:1 ratio, {len(compressed)} bytes")
            
        except Exception as e:
            print(f"  Chunk size {chunk_desc:>10}: Error - {type(e).__name__}")
            raise
    
    # Shuffle and bit-shuffle testing
    print(f"\nTesting shuffle algorithms:")
    
    # Create data with patterns that benefit from shuffling
    structured_data = np.array([
        (i, i*2, i*3, i%256) for i in range(25000)
    ], dtype=[('a', 'i4'), ('b', 'i4'), ('c', 'i4'), ('d', 'u1')])
    structured_bytes = structured_data.tobytes()
    
    shuffle_types = [
        (blosc.NOSHUFFLE, "No shuffle"),
        (blosc.SHUFFLE, "Byte shuffle"),
        (blosc.BITSHUFFLE, "Bit shuffle")
    ]
    
    for shuffle_type, shuffle_name in shuffle_types:
        try:
            compressed = blosc.compress(structured_bytes, typesize=4, shuffle=shuffle_type)
            ratio = len(structured_bytes) / len(compressed)
            # Each shuffle filter must undo itself exactly
            assert blosc.decompress(compressed) == structured_bytes, f"{shuffle_name} round-trip lost data"
            print(f"  {shuffle_name:>12}: {ratio:6.2f}:1 ratio")
            
        except Exception as e:
            print(f"  {shuffle_name:>12}: Error - {type(e).__name__}")
            raise
    
    # Compression metadata inspection
    print(f"\nCompression metadata inspection:")
    
    # Compress with specific settings
    test_compressed = blosc.compress(
        array_bytes[:10000],  # First 10KB
        typesize=4,
        clevel=5,
        shuffle=blosc.SHUFFLE,
        cname='lz4'
    )
    
    # Get compression info (returns string, not tuple)
    info = blosc.get_clib(test_compressed)
    print(f"Compression library: {info}")
    
    # Get buffer sizes (this returns tuple)
    nbytes, cbytes, blocksize = blosc.get_cbuffer_sizes(test_compressed)
    assert info == 'LZ4', info
    assert nbytes == 10000 and cbytes == len(test_compressed), (nbytes, cbytes, len(test_compressed))
    print(f"Compressed metadata:")
    print(f"  Uncompressed size: {nbytes} bytes")
    print(f"  Compressed size: {cbytes} bytes") 
    print(f"  Block size: {blocksize} bytes")
    
    # Memory usage optimization using memoryview instead of array_interface
    print(f"\nMemory usage optimization:")
    
    # Pre-allocate buffer for decompression
    original_size = len(array_bytes[:10000])
    output_buffer = bytearray(original_size)
    
    # Decompress into the pre-allocated buffer.
    decompressed_data = blosc.decompress(test_compressed)
    output_buffer[:len(decompressed_data)] = decompressed_data
    print(f"Decompressed using regular decompress: {len(decompressed_data)} bytes")
    
    # Verify decompression worked: the buffer must hold exactly the 10,000 source bytes
    assert bytes(output_buffer) == array_bytes[:10000], "blosc buffer decompression does not match the source bytes"
    print("✓ Buffer decompression successful")
    
    # Global settings management
    print(f"\nGlobal settings:")
    print(f"Current threads: {blosc.nthreads}")
    
    # Reset to defaults
    blosc.set_nthreads(1)
    assert blosc.nthreads == 1, blosc.nthreads
    print(f"Reset threads: {blosc.nthreads}")
    
    print("✓ Advanced Blosc features and chunking completed")
    print("blosc example 3 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"blosc example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: blosc functionality test 4

In [ ]:
import blosc
import numpy as np
import pickle
import json

# Example 4: Data serialization and real-world compression scenarios
print("Example 4: Data serialization and real-world compression scenarios")

try:
    print("Testing real-world data compression scenarios:")
    
    # Scenario 1: Scientific data (time series)
    print("\n1. Scientific time series data:")
    
    # Generate realistic sensor data with noise
    time_points = np.linspace(0, 100, 50000)
    sensor_data = {
        'timestamp': time_points,
        'temperature': 20 + 5 * np.sin(time_points * 0.1) + np.random.normal(0, 0.5, 50000),
        'pressure': 1013 + 10 * np.cos(time_points * 0.05) + np.random.normal(0, 1, 50000),
        'humidity': 50 + 20 * np.sin(time_points * 0.08) + np.random.normal(0, 2, 50000)
    }
    
    # Serialize as structured array
    structured_array = np.array(list(zip(
        sensor_data['timestamp'],
        sensor_data['temperature'], 
        sensor_data['pressure'],
        sensor_data['humidity']
    )), dtype=[('time', 'f8'), ('temp', 'f4'), ('pressure', 'f4'), ('humidity', 'f4')])
    
    structured_bytes = structured_array.tobytes()
    
    # Compress scientific data
    sci_compressed = blosc.compress(structured_bytes, typesize=4, shuffle=blosc.SHUFFLE, cname='lz4')
    sci_ratio = len(structured_bytes) / len(sci_compressed)
    
    print(f"  Original size: {len(structured_bytes)} bytes")
    print(f"  Compressed size: {len(sci_compressed)} bytes")
    print(f"  Compression ratio: {sci_ratio:.2f}:1")
    
    # Scenario 2: Image-like data (2D arrays)
    print("\n2. Image-like 2D array data:")
    
    # Create synthetic image data
    image_data = np.zeros((512, 512), dtype=np.uint8)
    
    # Add some patterns (more compressible than random)
    for i in range(0, 512, 64):
        for j in range(0, 512, 64):
            image_data[i:i+32, j:j+32] = (i + j) % 256
    
    image_bytes = image_data.tobytes()
    
    # Test different approaches for image data
    approaches = [
        ('no_shuffle', blosc.NOSHUFFLE),
        ('byte_shuffle', blosc.SHUFFLE), 
        ('bit_shuffle', blosc.BITSHUFFLE)
    ]
    
    for approach_name, shuffle_type in approaches:
        img_compressed = blosc.compress(image_bytes, typesize=1, shuffle=shuffle_type, cname='lz4')
        img_ratio = len(image_bytes) / len(img_compressed)
        assert blosc.decompress(img_compressed) == image_bytes, f"{approach_name} image round-trip lost data"
        print(f"  {approach_name:>12}: {img_ratio:6.2f}:1 ratio")
    
    # Scenario 3: JSON-like structured data
    print("\n3. JSON-like structured data:")
    
    # Create nested data structure
    json_data = {
        'users': [
            {
                'id': i,
                'name': f'user_{i:04d}',
                'email': f'user{i}@example.com',
                'score': i * 10.5,
                'active': i % 3 == 0,
                'metadata': {'level': i % 10, 'points': i * 100}
            }
            for i in range(5000)
        ],
        'timestamp': '2024-01-01T00:00:00Z',
        'version': '1.0'
    }
    
    # Compare different serialization + compression approaches
    serialization_methods = [
        ('json', lambda x: json.dumps(x).encode('utf-8')),
        ('pickle', lambda x: pickle.dumps(x))
    ]
    
    for method_name, serialize_func in serialization_methods:
        try:
            # Serialize data
            serialized = serialize_func(json_data)
            
            # Compress serialized data
            compressed = blosc.compress(serialized, shuffle=blosc.SHUFFLE, cname='lz4')
            
            # Calculate ratios
            ratio = len(serialized) / len(compressed)
            
            print(f"  {method_name:>8}: {len(serialized):>8} -> {len(compressed):>8} bytes ({ratio:.2f}:1)")
            
        except Exception as e:
            print(f"  {method_name:>8}: Error - {type(e).__name__}")
            raise
    
    # Scenario 4: Database-like rows
    print("\n4. Database-like row data:")
    
    # Simulate database table with repetitive patterns
    db_records = []
    categories = ['A', 'B', 'C', 'D', 'E']
    statuses = ['active', 'inactive', 'pending']
    
    for i in range(10000):
        record = {
            'id': i,
            'category': categories[i % len(categories)],
            'status': statuses[i % len(statuses)],
            'value': round(i * 1.23, 2),
            'created_date': f'2024-{(i%12)+1:02d}-{(i%28)+1:02d}'
        }
        db_records.append(record)
    
    # Convert to bytes (simulate row-based storage)
    db_serialized = pickle.dumps(db_records)
    
    # Test compression
    db_compressed = blosc.compress(db_serialized, shuffle=blosc.SHUFFLE, cname='lz4')
    db_ratio = len(db_serialized) / len(db_compressed)
    
    print(f"  Original size: {len(db_serialized)} bytes")
    print(f"  Compressed size: {len(db_compressed)} bytes") 
    print(f"  Compression ratio: {db_ratio:.2f}:1")
    
    # Decompression verification
    print(f"\nVerifying decompression integrity:")
    
    test_cases = [
        ('Scientific data', structured_bytes, sci_compressed),
        ('Image data', image_bytes, blosc.compress(image_bytes, typesize=1, shuffle=blosc.SHUFFLE)),
        ('Database data', db_serialized, db_compressed)
    ]
    
    for case_name, original, compressed in test_cases:
        try:
            decompressed = blosc.decompress(compressed)
            integrity_ok = original == decompressed
            print(f"  {case_name:>15}: {'✓ OK' if integrity_ok else '✗ FAIL'}")
            assert integrity_ok, f"{case_name} round-trip lost data"
            
        except Exception as e:
            print(f"  {case_name:>15}: ✗ Error - {type(e).__name__}")
            raise
    
    print("✓ Data serialization and real-world compression scenarios completed")
    print("blosc example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"blosc example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: blosc functionality test 5

In [ ]:
import blosc
import numpy as np

# Example 5: Blosc package capabilities and high-performance data compression
print("Example 5: Blosc package capabilities and high-performance data compression")

try:
    print("Blosc - High-performance data compression library")
    
    # Show package version and build info (using correct attributes)
    print(f"Blosc version: {blosc.__version__}")
    print(f"Blosc library version: {blosc.VERSION_STRING}")
    print(f"Blosc date: {blosc.VERSION_DATE}")
    
    # Key Blosc capabilities
    print("\nKey Blosc capabilities:")
    capabilities = [
        "⚡ Extremely fast compression and decompression",
        "🧵 Multi-threaded compression for modern CPUs", 
        "🔄 Multiple compression algorithms (LZ4, Snappy, Zlib, Zstd)",
        "🎲 Byte and bit shuffling for better compression ratios",
        "🎯 Optimized for numerical and scientific data",
        "💾 Memory-efficient streaming and chunking",
        "🔧 Fine-tuned control over compression parameters",
        "🏗️ Built for HDF5, NumPy, and data science workflows"
    ]
    
    for capability in capabilities:
        print(f"  {capability}")
    
    # Available compression algorithms (using correct function)
    compressors = blosc.compressor_list()
    print(f"\nAvailable compression algorithms:")
    for i, compressor in enumerate(compressors):
        print(f"  {i+1}. {compressor}")
    
    # Performance characteristics
    print(f"\nPerformance characteristics:")
    performance_features = [
        ("Speed", "Designed for speed-first compression"),
        ("Threading", "Automatic multi-core utilization"),
        ("SIMD", "Vectorized operations on modern CPUs"),
        ("Cache-friendly", "Optimized memory access patterns"),
        ("Chunking", "Efficient processing of large datasets"),
        ("Zero-copy", "Memory-efficient decompression options")
    ]
    
    for feature, description in performance_features:
        print(f"  • {feature}: {description}")
    
    # Common use cases
    print("\nCommon Blosc use cases:")
    use_cases = [
        "🔬 Scientific computing and research data",
        "📊 Time series and sensor data compression", 
        "🖼️ Image and signal processing pipelines",
        "💾 Database storage engine optimization",
        "🌐 Network data transmission acceleration",
        "📈 Financial market data processing",
        "🧬 Bioinformatics and genomics data",
        "🏭 Industrial IoT and monitoring systems"
    ]
    
    for use_case in use_cases:
        print(f"  {use_case}")
    
    # Integration ecosystem
    print("\nIntegration ecosystem:")
    integrations = [
        "• HDF5: Native Blosc filter support",
        "• NumPy: Array compression and storage", 
        "• Pandas: DataFrame compression backends",
        "• PyTables: High-performance data storage",
        "• Zarr: Cloud-native array storage format",
        "• Dask: Distributed computing compression",
        "• Apache Arrow: Columnar data compression",
        "• Custom applications: Direct API usage"
    ]
    
    for integration in integrations:
        print(f"  {integration}")
    
    # Compression strategy guidance
    print("\nCompression strategy guidance:")
    strategies = [
        "🎯 Use LZ4 for maximum speed with decent compression",
        "📦 Use Zstd for best compression ratios", 
        "⚡ Enable shuffling for structured/numerical data",
        "🧵 Use multiple threads for large datasets",
        "🔧 Tune chunk size based on data access patterns",
        "💾 Consider memory vs compression ratio tradeoffs",
        "📊 Profile different algorithms with your data",
        "🎨 Use bit-shuffle for floating-point arrays"
    ]
    
    for strategy in strategies:
        print(f"  {strategy}")
    
    # Demonstration of key features
    print(f"\nKey features demonstration:")
    
    # 1. Multi-algorithm comparison
    test_data = np.random.rand(50000).astype(np.float32).tobytes()
    
    print(f"  Algorithm comparison ({len(test_data)} bytes):")
    for compressor in compressors[:3]:  # Test first 3
        try:
            compressed = blosc.compress(test_data, cname=compressor)
            ratio = len(test_data) / len(compressed)
            assert blosc.decompress(compressed) == test_data, f"{compressor} round-trip lost data"
            print(f"    {compressor:>8}: {ratio:5.2f}:1 ratio")
        except:
            print(f"    {compressor:>8}: Not available")
            raise
    
    # 2. Threading demonstration
    print(f"\n  Threading impact:")
    original_threads = blosc.nthreads
    
    for threads in [1, 2]:
        blosc.set_nthreads(threads)
        import time
        start = time.time()
        compressed = blosc.compress(test_data, cname='lz4')
        elapsed = time.time() - start
        print(f"    {threads} thread{'s' if threads > 1 else ' '}: {elapsed*1000:.1f}ms")
    
    blosc.set_nthreads(original_threads)  # Reset
    
    # 3. Shuffle effectiveness
    print(f"\n  Shuffle effectiveness:")
    structured_data = np.array([(i, i*2, i*3) for i in range(10000)], 
                              dtype=[('a', 'i4'), ('b', 'i4'), ('c', 'i4')]).tobytes()
    
    no_shuffle = blosc.compress(structured_data, shuffle=blosc.NOSHUFFLE)
    with_shuffle = blosc.compress(structured_data, shuffle=blosc.SHUFFLE)
    
    # Integer columns that grow slowly share their high bytes, so shuffling must compress better
    assert len(with_shuffle) < len(no_shuffle), (len(with_shuffle), len(no_shuffle))
    assert blosc.decompress(with_shuffle) == structured_data, "shuffled round-trip lost data"
    print(f"    No shuffle: {len(structured_data)/len(no_shuffle):.2f}:1")
    print(f"    With shuffle: {len(structured_data)/len(with_shuffle):.2f}:1")
    
    # Best practices summary
    print(f"\nBlosc best practices:")
    best_practices = [
        "📏 Choose chunk size based on L2/L3 cache size",
        "🎯 Match compressor to data characteristics", 
        "🧵 Scale threads with available CPU cores",
        "🔄 Enable appropriate shuffle for data type",
        "⚖️ Balance compression ratio vs speed needs",
        "💾 Use memory mapping for very large datasets",
        "📊 Benchmark with representative data samples",
        "🔧 Consider compression level tuning for optimal results"
    ]
    
    for practice in best_practices:
        print(f"  {practice}")
    
    # Configuration summary
    print(f"\nCurrent Blosc configuration:")
    print(f"  Version: {blosc.__version__}")
    print(f"  Default compressor: {blosc.cnames[0]}")  # First available compressor
    print(f"  Threads: {blosc.nthreads}")
    print(f"  Available compressors: {len(compressors)}")
    
    print(f"\nBlosc provides world-class compression performance")
    print("Essential for high-performance data processing pipelines")
    print("blosc example 5 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"blosc example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")